# Clean the Gutenberg Poetry Corpus
Steps:
1. remove translations
2. remove prose (introductions, notes, essays)
3. fix small text problems (HTML codes, the "0" → "O" slip)
4. add the author of each book, and check it against public-domain-poetry
5. remove duplicate books and lines
6. join the lines back into books and save

Output: `data/gutenberg_clean.parquet`. Run `clean_pdp.ipynb` first: step 4 uses its output.

In [1]:
import html
import re
import pandas as pd

DATA = "../data/"

lines = pd.read_parquet(DATA + "gutenberg-poetry-corpus/data/train-00000-of-00001-fa9fb9e1f16eed7e.parquet")
catalog = pd.read_csv(DATA + "gutenberg-catalog/pg_catalog.csv", low_memory=False)
catalog["Text#"] = pd.to_numeric(catalog["Text#"], errors="coerce")
catalog = catalog.set_index("Text#")

books = catalog.loc[lines["gutenberg_id"].unique(), ["Title", "Authors", "LoCC"]].fillna("")
books["Title"] = books["Title"].str.replace(r"\s+", " ", regex=True)

print(len(lines), "lines in", len(books), "books")

3085117 lines in 1191 books


## 1. Remove translations
Like in public-domain-poetry, translations show the translator's style, not the poet's.
The catalogue's library class (`LoCC`) tells us which literature a book belongs to. We drop the classes
for foreign literature, and any book with a `[Translator]` in its author list. One more translation
isn't marked either way, so it's added by hand.

In [2]:
FOREIGN = [
    "PA",  # Greek and Latin (Homer, Virgil, Ovid, ...)
    "PQ",  # French, Italian, Spanish (Dante, Ariosto, ...)
    "PT",  # German and Scandinavian (Goethe, ...)
    "PK",  # Indian (Ramayana, ...)
    "PH",  # Finnish (Kalevala)
    "PG",  # Slavic
    "PL",  # East Asian
    "PB",  # Celtic
    "PJ",  # Arabic, Hebrew, ...
    "PM",  # other languages
]

foreign = books["LoCC"].str[:2].isin(FOREIGN)
translator = books["Authors"].str.contains(r"\[Translator\]")
by_hand = books.index.isin([24815])  # "Translations of German Poetry in American Magazines"
translations = books.index[foreign | translator | by_hand]

before = len(lines)
lines = lines[~lines["gutenberg_id"].isin(translations)]
print("removed", len(translations), "translated books,", before - len(lines), "lines")

removed 190 translated books, 775271 lines


## 2. Remove prose
Many books have a prose introduction, notes or whole essays. Gutenberg wraps prose at about 65
characters, so prose lines are as short as verse lines. But a wrapped prose line often starts in the
middle of a sentence, with a small letter. Verse lines almost never do.

So for every line we look at the 11 lines around it. If at least 40% of them start with a small letter
**and** they are 50+ characters long on average, it's prose. Free verse with small letters (H.D., William
Carlos Williams) has short lines, so it stays. The window misses about 5 lines at each edge of a prose
part, so we also drop the 5 lines on each side.

A book that's mostly prose (essays, criticism) is dropped completely.

Known side effect: this Gutenberg edition of Whitman wraps his long lines, so about 3% of his lines here
look like prose and get dropped.

In [3]:
def around(s):
    # the 11 lines around each line, within one book
    return s.rolling(11, center=True, min_periods=1)


lines["small_letter"] = lines["line"].str.match(r"\s*[a-z]").astype(float)
lines["length"] = lines["line"].str.len()

by_book = lines.groupby("gutenberg_id")
small_share = by_book["small_letter"].transform(lambda s: around(s).mean())
avg_length = by_book["length"].transform(lambda s: around(s).mean())
lines["prose"] = ((small_share >= 0.4) & (avg_length >= 50)).astype(float)

# widen by 5 lines on each side
lines["prose"] = lines.groupby("gutenberg_id")["prose"].transform(lambda s: around(s).max()) > 0
print(lines["prose"].sum(), "prose lines")

# books that are mostly prose
prose_share = lines.groupby("gutenberg_id")["prose"].mean()
prose_books = prose_share.index[prose_share > 0.5]
print(len(prose_books), "books are mostly prose, e.g.:")
print(books.loc[prose_books[:6], "Title"].tolist())

lines = lines[~lines["prose"] & ~lines["gutenberg_id"].isin(prose_books)]
print(len(lines), "lines left")

50577 prose lines
22 books are mostly prose, e.g.:
['Flower Fables', 'Codex Junius 11', 'The Grey Brethren, and Other Fragments in Prose and Verse', 'First Project Gutenberg Collection of Edgar Allan Poe', 'The Flower of the Mind', 'Discourses on Satire and on Epic Poetry']
2254157 lines left


Check: *Hiawatha* (book 19) started with 23 lines of prose about the legends. Now only its last short
line is left, followed by the verse it quotes.

In [4]:
lines[lines["gutenberg_id"] == 19]["line"].head(5).tolist()

['Hiawatha to me, especially:',
 '"Wah-wah-taysee, little fire-fly,',
 'Little, flitting, white-fire insect',
 'Little, dancing, white-fire creature,',
 'Light me with your little candle,']

## 3. Fix small text problems
A few lines have HTML codes like `&eacute;` (é), and some have the same OCR slip as public-domain-poetry
("Strive, 0 heart"). Same fix as there: a "0" at the end of a line is deleted, any other stand-alone "0"
becomes "O".

In [5]:
lines["line"] = lines["line"].map(html.unescape)

END_ZERO = r"(?<!\d)[ \xa0]*\b0[ \xa0]*$"
ZERO = r"(?<![\d.,])\b0\b(?![.,]?\d)"
print(lines["line"].str.contains(END_ZERO).sum(), "lines end in a stray 0")
print(lines["line"].str.contains(ZERO).sum(), "lines have the 0 slip")

lines["line"] = lines["line"].str.replace(END_ZERO, "", regex=True)
lines["line"] = lines["line"].str.replace(ZERO, "O", regex=True)

0 lines end in a stray 0


19 lines have the 0 slip


## 4. Add the author of each book
Same idea as in `author_counts.ipynb`: the catalogue writes `Dickinson, Emily, 1830-1886`, which becomes
`Emily Dickinson`. Books with more than one author get `Unknown`.

The same poet should have the same name in both corpora, but they're often written differently
("Robert W. Service" here, "Robert William Service" in public-domain-poetry). So if exactly one poet in
public-domain-poetry has the same first and last name, we use that spelling.

In [6]:
def parse_author(authors):
    people = [a.strip().replace(" [Author]", "") for a in authors.split(";")]
    people = [a for a in people if "[" not in a]  # drop editors, illustrators, ...
    if len(people) != 1:
        return "Unknown"
    names = [re.sub(r"\(.*?\)", "", p).strip() for p in people[0].split(", ") if not re.search(r"\d", p)]
    if not names or names[0] in ["Various", "Anonymous", "Unknown"]:
        return "Unknown"
    last = names[0]
    first = names[1] if len(names) > 1 else ""
    if first.endswith(last):  # "Tennyson, Alfred Tennyson, Baron" -> "Alfred Tennyson"
        return first
    return (first + " " + last).strip()


pdp = pd.read_parquet(DATA + "pdp_clean.parquet")
pdp_authors = set(pdp["author"])


def first_last(name):
    words = name.split()
    return (words[0] + " " + words[-1]).lower()


pdp_by_first_last = {}
for a in pdp_authors:
    pdp_by_first_last.setdefault(first_last(a), []).append(a)


def same_as_pdp(name):
    if name == "Unknown" or name in pdp_authors:
        return name
    matches = pdp_by_first_last.get(first_last(name), [])
    return matches[0] if len(matches) == 1 else name


books["author"] = books["Authors"].map(parse_author).map(same_as_pdp)
print((books["author"] != "Unknown").sum(), "books with a named author")

975 books with a named author


### Check the authors against public-domain-poetry
The catalogue isn't always right about who wrote the poems in a book:
- some anthologies are credited to the person who put them together (*The Home Book of Verse* by
  Burton Stevenson holds poems by hundreds of poets)
- some poets are under a different name ("A. E. Housman" here, "Alfred Edward Housman" there; *The City of
  Dreadful Night* is by the second James Thomson, not the first)

public-domain-poetry knows the author of each poem, so for every book we look up who wrote its lines there:
- if at least half of the book is found there, and 80%+ of that is by one poet → that poet is the author
- if the lines come from 5+ poets and the credited author wrote less than half of them → it's an
  anthology, so `Unknown`. Only if at least 15% of the book is found, so a poet's book that quotes other
  poets in its notes (like Gray's *Select Poems*, 10% found) keeps its author.

Three books of quotations that the rule misses are set to `Unknown` by hand.

In [7]:
def clean_text(s):
    # lowercase, letters only, so small differences don't matter
    return s.str.lower().str.replace(r"[^a-z ]", " ", regex=True).str.split().str.join(" ")


lines["clean"] = clean_text(lines["line"])

named_pdp = pdp[pdp["author"] != "Unknown"]
pdp_lines = named_pdp[["author"]].assign(line=named_pdp["text"].str.split("\n")).explode("line")
pdp_lines["clean"] = clean_text(pdp_lines["line"])
pdp_lines = pdp_lines[pdp_lines["clean"].str.len() >= 25].drop_duplicates("clean")

found = lines[["gutenberg_id", "clean"]].merge(pdp_lines[["clean", "author"]], on="clean")
stats = found.groupby("gutenberg_id")["author"].agg(found="size", poets="nunique")
stats["top_poet"] = found.groupby("gutenberg_id")["author"].agg(lambda a: a.value_counts().index[0])
stats["top_share"] = found.groupby("gutenberg_id")["author"].agg(lambda a: a.value_counts(normalize=True).iloc[0])
stats["share_of_book"] = stats["found"] / lines.groupby("gutenberg_id").size()

# one poet wrote (almost) everything we found
one_poet = (stats["found"] >= 30) & (stats["share_of_book"] >= 0.5) & (stats["top_share"] >= 0.8)
new_name = stats.loc[one_poet, "top_poet"]
renamed = new_name[new_name != books.loc[new_name.index, "author"]]
print(len(renamed), "books get a different author, e.g.:")
print(pd.DataFrame({"catalogue": books.loc[renamed.index, "author"], "now": renamed}).head(8))
books.loc[renamed.index, "author"] = renamed

51 books get a different author, e.g.:
                  catalogue                    now
213          A. B. Paterson         Banjo Paterson
263            Joyce Kilmer    Alfred Joyce Kilmer
264            Joyce Kilmer    Alfred Joyce Kilmer
304          A. B. Paterson         Banjo Paterson
312   Stephen Vincent Benét  Stephen Vincent Benet
1238          James Thomson   James Thomson (B.V.)
1317         A. B. Paterson         Banjo Paterson
1333           R. F. Murray   Robert Fuller Murray


In [8]:
# many poets, and the credited author wrote less than half
found["book_author"] = found["gutenberg_id"].map(books["author"])
stats["own_share"] = (found["author"] == found["book_author"]).groupby(found["gutenberg_id"]).mean()
many_poets = (stats["found"] >= 30) & (stats["share_of_book"] >= 0.15) & (stats["poets"] >= 5) & (stats["own_share"] < 0.5)

anthologies = [b for b in stats.index[many_poets] if books.at[b, "author"] != "Unknown"]
anthologies += [12286, 13983, 25631]  # by hand: flower quotations, The Book of the Epic, A Poetical Cook-Book
books.loc[anthologies, "author"] = "Unknown"
print(len(anthologies), "books are anthologies, now Unknown:")
print(books.loc[anthologies, "Title"].str[:45].tolist())

18 books are anthologies, now Unknown:
['A Collection of Ballads', 'The Home Book of Verse — Volume 1', 'The Home Book of Verse — Volume 2', 'The Home Book of Verse — Volume 3', 'The Home Book of Verse — Volume 4', 'Biographia Literaria', "The Poet's Poet : essays on the character and", 'In The Yule-Log Glow, Book IV', 'Six Centuries of English Poetry: Tennyson to ', 'Language of Flowers', 'English Verse: Specimens Illustrating its Pri', 'The Bridling of Pegasus: Prose Papers on Poet', "A Key to Lord Tennyson's 'In Memoriam'", 'Oxford Lectures on Poetry', 'Studies of Contemporary Poets', 'Flowers and Flower-Gardens With an Appendix o', "The Book of the Epic: The World's Great Epics", 'A Poetical Cook-Book']


## 5. Remove duplicate books and lines
Some books are in the corpus more than once: two editions of *Paradise Lost*, three of *Hiawatha*, Dickinson's
three series *and* the complete edition, and anthologies full of poems that are also in the poets' own books.
Duplicates would end up in both the training and the test set.

We go through the books from biggest to smallest, books with a known author first (so a poem keeps its
author rather than "Unknown"). A book is dropped if more than half of its lines were already seen in an
earlier book. Very short lines (under 20 letters) are ignored, since they repeat a lot anyway.

Anthologies that are less than half duplicate stay, but the poems they share with other books would still
be there twice. So after that, we also remove each line that already appeared in an *earlier* book.
Lines repeated inside one book (refrains) stay.

In [9]:
book_lines = lines.groupby("gutenberg_id")["clean"].apply(lambda s: set(c for c in s if len(c) >= 20))

order = books.loc[book_lines.index].assign(size=lines.groupby("gutenberg_id").size(), unknown=lambda b: b["author"] == "Unknown")
order = order.sort_values(["unknown", "size"], ascending=[True, False]).index

seen = set()
duplicates = []
for book in order:
    book_set = book_lines[book]
    if book_set and len(book_set & seen) / len(book_set) > 0.5:
        duplicates.append(book)
    else:
        seen |= book_set

before = len(lines)
lines = lines[~lines["gutenberg_id"].isin(duplicates)]
print("removed", len(duplicates), "duplicate books,", before - len(lines), "lines, e.g.:")
print(books.loc[duplicates[:8], "Title"].str[:50].tolist())

# lines that already appeared in an earlier book
lines["order"] = lines["gutenberg_id"].map(pd.Series(range(len(order)), index=order))
first_book = lines.groupby("clean")["order"].transform("min")
repeated = (lines["clean"].str.len() >= 20) & (lines["order"] > first_book)

lines = lines[~repeated]
print("removed", repeated.sum(), "lines that were already in an earlier book")

removed 106 duplicate books, 212192 lines, e.g.:
['Poems and Songs of Robert Burns', 'Paradise Lost', 'The Song of Hiawatha: An Epic Poem', 'The Song of Hiawatha', 'The Story of Hiawatha, Adapted from Longfellow', 'A Selection from the Lyrical Poems of Robert Herri', 'Maurine and Other Poems', 'Tales of a Wayside Inn']


removed 111811 lines that were already in an earlier book


## 6. Join the lines into books and save

In [10]:
text = lines.groupby("gutenberg_id")["line"].apply("\n".join)

clean = books.loc[text.index, ["author", "Title"]].rename(columns={"Title": "title"})
clean["text"] = text
clean["n_lines"] = lines.groupby("gutenberg_id").size()
clean = clean[clean["n_lines"] >= 4]

clean.to_parquet(DATA + "gutenberg_clean.parquet")
print(len(clean), "books,", clean["n_lines"].sum(), "lines,", clean["author"].nunique(), "authors")
print(round((clean["author"] == "Unknown").mean() * 100), "% of books are Unknown (anthologies)")

869 books, 1930147 lines, 485 authors
17 % of books are Unknown (anthologies)


### Candidate authors after cleaning

In [11]:
candidates = ["Emily Dickinson", "Christina Rossetti", "Edgar Allan Poe", "John Keats", "Walt Whitman", "John Milton"]
clean[clean["author"].isin(candidates)].groupby("author").agg(books=("text", "size"), lines=("n_lines", "sum"))

,books,lines
author,,
Edgar Allan Poe,1,3651
Emily Dickinson,1,5392
John Keats,2,7145
John Milton,4,21092
Walt Whitman,1,7437


Rossetti has no books in this corpus, so she's not in the table.